In [15]:
!pip install tensorflow

In [124]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import HeNormal
from sklearn.metrics import classification_report
df = pd.read_csv("recidivism_merged_dataset.csv")


In [109]:
# 1. Arrest Rate: How 'busy' has their criminal history been relative to their age?
df['Arrest_Rate'] = df['Prior_Arrests_Count'] / (df['Age'] + 1)

# 2. Felony Ratio: What percentage of their arrests were serious?
# Use a small epsilon (1e-5) to avoid dividing by zero
df['Felony_Ratio'] = df['Prior_Felony_Arrests'] / (df['Prior_Arrests_Count'] + 1e-5)

# 3. Young Offender Flag: Research shows age at first offense is a huge predictor
df['Is_Young_Offender'] = (df['Age'] < 25).astype(int)
# Weights: Violent=3x, Felony=2x, Misdemeanor=1x
df['Criminal_Severity_Score'] = (df['Prior_Violent_Arrests'] * 3) + \
                                (df['Prior_Felony_Arrests'] * 2) + \
                                df['Prior_Misdemeanor_Arrests']
# If Employed is 1 (Yes), (1 - Employed) becomes 0.
# If they are unemployed (0) AND have substance abuse (1), the index hits 2 (Maximum Instability)
df['Instability_Index'] = (1 - df['Employed']) + df['Substance_Abuse']
# How many years they served per arrest on average
df['Sentence_per_Arrest'] = df['Sentence_Years'] / (df['Prior_Arrests_Count'] + 1)
# Number of felonies relative to their total lifespan
df['Felony_Density'] = df['Prior_Felony_Arrests'] / (df['Age'] + 1)

In [110]:
features = [
    "Age",
    "Prior_Arrests_Count",
    "Prior_Felony_Arrests",
    "Prior_Misdemeanor_Arrests",
    "Prior_Violent_Arrests",
    "Sentence_Years",
    "Employed",
    "Substance_Abuse",
    "Gender_Encoded",
    "Offense_Type_Encoded",
    "Education_Level_Encoded",
    
    # Your Previous Feature Engineering
    "Arrest_Rate",
    "Felony_Ratio",
    "Is_Young_Offender",
    
    # --- YOUR NEW FEATURES ---
    "Criminal_Severity_Score",
    "Instability_Index",
    "Sentence_per_Arrest",
    "Felony_Density"
]

X = df[features]
Y = df["Recidivism"]


In [111]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [112]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [113]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(X.shape[1],),kernel_initializer=HeNormal()),
    Dropout(0.2),
    Dense(32,activation='relu',kernel_initializer=HeNormal()),
    Dropout(0.2),
    Dense(1, activation='sigmoid')])

        
          

C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [114]:
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])


In [119]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=50,          # Stop if no improvement for 5 epochs in a row
    restore_best_weights=True # Keep the best version of the model
)

In [120]:
history=model.fit(X_train,Y_train,
                  epochs=400,
                  batch_size=32,
                  validation_split=0.2,
                 callbacks=[early_stop],
                 verbose=1)  


Epoch 1/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6844 - loss: 0.5954 - val_accuracy: 0.6678 - val_loss: 0.6052
Epoch 2/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6851 - loss: 0.5971 - val_accuracy: 0.6678 - val_loss: 0.6053
Epoch 3/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6832 - loss: 0.5971 - val_accuracy: 0.6674 - val_loss: 0.6053
Epoch 4/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6821 - loss: 0.5958 - val_accuracy: 0.6682 - val_loss: 0.6054
Epoch 5/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6832 - loss: 0.5976 - val_accuracy: 0.6686 - val_loss: 0.6056
Epoch 6/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6865 - loss: 0.5948 - val_accuracy: 0.6674 - val_loss: 0.6054
Epoch 7/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6838 - loss: 0.5972 - val_accuracy: 0.6671 - val_loss: 0.6055
Epoch 8/400
963/963 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6833 - loss: 0.5974 - val_accu

In [121]:
print(df['Recidivism'].value_counts(normalize=True))

Recidivism
0    0.526553
1    0.473447
Name: proportion, dtype: float64


In [122]:
Y_prob=model.predict(X_test)
Y_pred = (Y_prob > 0.5).astype(int)

301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  


In [123]:
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.70      0.70      0.70      5089
           1       0.67      0.66      0.67      4541

    accuracy                           0.68      9630
   macro avg       0.68      0.68      0.68      9630
weighted avg       0.68      0.68      0.68      9630

